# 第三部分：RAII 与资源生命周期

## 实验 4：提前返回与自动清理

实验 3 说明局部 RAII 对象会在离开作用域时析构。`return` 正是离开当前函数作用域的一种方式，因此函数不需要在每个返回分支重复释放资源。

本实验从单个提前返回扩展到多个校验分支和多个资源，观察不同返回位置究竟会销毁哪些对象。所有文件输出都位于 `outputs/04/`。

核心规则：

> 执行 `return` 时，当前作用域中已经构造完成的自动存储期对象会按构造顺序的逆序析构。尚未执行到构造位置的对象不存在，也不需要清理。

这使资源清理与业务分支解耦：业务代码决定何时返回，资源类型决定如何释放。

In [1]:
#include <cstdio>
#include <filesystem>
#include <iostream>
#include <stdexcept>
#include <string>

### 1. 准备可观察的 RAII 文件类

`TrackedFile` 在构造和析构时输出标签，使每个返回分支的资源生命周期都可以直接观察。

In [2]:
std::filesystem::create_directories("outputs/04");

class TrackedFile
{
public:
    TrackedFile(
        const std::string &label,
        const std::string &path,
        const char *mode)
        : label_(label),
          path_(path),
          file_(std::fopen(path_.c_str(), mode))
    {
        if (file_ == nullptr)
        {
            throw std::runtime_error(
                std::string("cannot open file: ") + path_);
        }

        std::cout << label_ << " acquire" << std::endl;
    }

    ~TrackedFile()
    {
        std::cout << label_ << " release" << std::endl;
        std::fclose(file_);
    }

    TrackedFile(const TrackedFile &) = delete;
    TrackedFile &operator=(const TrackedFile &) = delete;

    std::FILE *get() const
    {
        return file_;
    }

private:
    std::string label_;
    std::string path_;
    std::FILE *file_;
};

### 2. 单个提前返回

函数写入开头后提前返回。代码没有显式调用 `std::fclose()`，但 `output` 的作用域仍然在 `return` 时结束。

In [3]:
void process_with_early_return(bool error)
{
    TrackedFile output(
        "output",
        "outputs/04/output.txt",
        "w");

    std::fputs("begin", output.get());

    if (error)
    {
        std::cout << "early return" << std::endl;
        return;
    }

    std::fputs("end", output.get());
}

process_with_early_return(true);

output acquire
early return
output release


预期输出：

```text
output acquire
early return
output release
```

`return` 跳过后面的普通语句，却不会跳过已构造局部对象的析构。

### 3. 在获取资源前使用 guard clause

RAII 能安全清理已获取的资源，但不代表应该无条件尽早获取资源。便宜且不依赖资源的校验应放在构造之前：无效请求直接返回，文件对象根本不会被创建。

In [4]:
bool write_validated_request(bool input_valid)
{
    if (!input_valid)
    {
        std::cout << "reject before acquire" << std::endl;
        return false;
    }

    TrackedFile output(
        "validated output",
        "outputs/04/validated.txt",
        "w");
    std::fputs("valid request", output.get());
    return true;
}

write_validated_request(false);

reject before acquire


这次输出中没有 `acquire` 或 `release`，因为控制流从未到达对象定义。C++ 只销毁已经完成构造的对象。

### 4. 多个返回点与多个资源

下面逐步获取输出文件和审计文件，并在不同阶段返回。用 `stop_after_step` 模拟业务失败位置。

In [5]:
bool build_report(int stop_after_step)
{
    TrackedFile report(
        "report",
        "outputs/04/report.txt",
        "w");
    std::fputs("report header", report.get());

    if (stop_after_step == 1)
    {
        std::cout << "stop after report" << std::endl;
        return false;
    }

    TrackedFile audit(
        "audit",
        "outputs/04/audit.txt",
        "w");
    std::fputs("audit entry", audit.get());

    if (stop_after_step == 2)
    {
        std::cout << "stop after audit" << std::endl;
        return false;
    }

    std::fputs("report body", report.get());
    return true;
}

先在只构造 `report` 后返回：

In [6]:
build_report(1);

report acquire
stop after report
report release


此时 `audit` 尚未构造，所以只会看到 `report release`。不存在的对象无需清理。

再在两个资源都构造完成后返回：

In [7]:
build_report(2);

report acquire
audit acquire
stop after audit
audit release
report release


这次 `audit` 和 `report` 都存在，因此返回时先释放后构造的 `audit`，再释放 `report`：

```text
report acquire
audit acquire
stop after audit
audit release
report release
```

清理集合由返回点之前已经完成构造的对象决定，清理顺序由对象生命周期规则决定。业务分支不需要手工维护这两件事。

### 5. 不必等到函数返回才释放

如果临时文件只在第一阶段需要，可以用内层作用域让它提前析构，随后继续执行函数的其他工作。

In [8]:
void process_in_stages()
{
    std::cout << "stage 1 begin" << std::endl;
    {
        TrackedFile temporary(
            "temporary",
            "outputs/04/temporary.txt",
            "w");
        std::fputs("temporary data", temporary.get());
    }

    std::cout << "stage 2 without temporary file" << std::endl;
}

process_in_stages();

stage 1 begin
temporary acquire
temporary release
stage 2 without temporary file


### 6. RAII 让控制流与资源管理解耦

手动管理时，每个业务分支都混入清理代码：

```text
if / return / error
        +
close A / close B / release C
```

使用 RAII 后职责更清晰：

```text
业务代码：决定 if、return 和结果
资源类型：决定如何释放
作用域规则：决定何时释放以及释放顺序
```

因此 guard clause 和提前返回不再天然增加资源泄漏风险，函数也能减少深层嵌套。

### 7. RAII 的边界与常见误区

- 普通 `return` 会销毁当前作用域中的局部对象。
- 异常传播也会进行栈展开并销毁已构造对象，这是实验 5 的主题。
- `std::exit()` 不会为当前调用栈执行普通局部对象的析构，因此不能把它当成一般返回。
- 进程崩溃、强制终止或断电时，不能假设析构函数一定执行。
- 如果用裸 `new` 创建 RAII 包装器再丢失指针，包装器本身不会析构；拥有者也必须由可靠的生命周期管理。

RAII 利用的是 C++ 正常的对象生命周期与栈展开规则，不是对任意终止方式都生效的魔法。

### 8. 与 Native SDK 的关系

SDK 函数往往包含参数校验、权限检查、Native API 调用和多种错误结果。若每个分支都手工释放句柄，新增一个返回点就可能引入泄漏。把句柄包装为局部 RAII 对象后，函数可以专注于返回正确的业务结果，类型系统负责释放已成功获取的资源。

### 实验结论

提前返回不会绕过自动对象的析构。返回时，只有已经构造完成的局部对象参与清理，并按构造顺序逆序析构。

RAII 因而允许使用清晰的 guard clause 和多个业务返回点，而无需复制资源释放代码；资源较早不再需要时，则应通过更小的内层作用域提前结束其生命周期。